# Задача

На основе самодельного автограда из 1 семинара нужно написать реализацию
* Module - базовый класс для всех объектов
* Neuron - класс, который соответствует одной строчке в линейном преобразовании
* Layer - Линейное преобразование + нелинейность
* MLP - целевая архитектура и граф вычислений для этого класса подсмотрите в классе TorchMLP в ячейках ниже

In [3]:
# ⚠️ Важно!
# Из ячейки не нужно убирать или добавлять другие импорты или объединять с тдругими ячейками
# Нельзя менять интерфейсы (входные аргументы)
# Это может сломать автотесты

import random

class Value:
    """Класс для автоматического дифференцирования."""

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad  # d(a+b)/da = 1
            other.grad += out.grad  # d(a+b)/db = 1
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad  # d(a*b)/da = b
            other.grad += self.data * out.grad  # d(a*b)/db = a
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad  # производная ReLU
        out._backward = _backward

        return out

    def backward(self):
        """Запускает backpropagation от этого узла."""
        # Топологическая сортировка
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        # Идем от выхода к входу и вычисляем градиенты
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    # TODO возможно, для этого класса не нужно переопределять еще другие магические __методы__ (__radd__ , __neg__, ...)


class Module():
    """
    Базовый класс для всех модулей.
    """

    def zero_grad(self):
        """Сбрасывает градиенты всех параметров в ноль.

        Проходит по параметрам, возвращаемым `parameters()`, и обнуляет `grad`.
        """
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        """Возвращает список параметров модуля.

        В базовом классе параметров нет, поэтому возвращается пустой список.
        """
        return []

    def state_dict(self):
        """Возвращает сериализованное состояние модуля.

        Должен быть реализован в наследниках.
        """
        return {}

    def load_state_dict(self, state_dict):
        """Загружает состояние модуля из `state_dict`.

        Должен быть реализован в наследниках.
        """
        pass


class Neuron(Module):
    """Нейрон соответствует одной строчке в матрице линейного преоразования
    """

    def __init__(self, n_in, nonlin=True):
        """Создает нейрон с заданным числом входов.

        Инициализирует веса и смещение, `nonlin` управляет применением ReLU.
        """
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        """Вычисляет выход нейрона для входного вектора `x`.

        Считает линейную комбинацию и опционально применяет ReLU.
        """
        if len(x) != len(self.w):
            raise ValueError(f"Expected input of size {len(self.w)}, got {len(x)}")

        out = self.b
        for wi, xi in zip(self.w, x):
            out = out + wi * xi

        return out.relu() if self.nonlin else out

    def parameters(self):
        """Возвращает список параметров нейрона (веса и смещение)."""
        return self.w + [self.b]

    def state_dict(self):
        """Возвращает состояние нейрона в виде словаря."""
        return {
            "w": [wi.data for wi in self.w],
            "b": self.b.data,
            "nonlin": self.nonlin,
        }

    def load_state_dict(self, state_dict):
        """Загружает состояние нейрона из `state_dict`.

        Проверяет размерность весов и присваивает значения весам и смещению.
        """
        if len(state_dict["w"]) != len(self.w):
            raise ValueError(
                f"Expected {len(self.w)} weights, got {len(state_dict['w'])}"
            )

        for wi, wi_new in zip(self.w, state_dict["w"]):
            wi.data = wi_new.data if isinstance(wi_new, Value) else wi_new

        b_new = state_dict["b"]
        self.b.data = b_new.data if isinstance(b_new, Value) else b_new

        if "nonlin" in state_dict:
            self.nonlin = state_dict["nonlin"]

    def __repr__(self):
        """Возвращает компактное строковое представление нейрона."""
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"


class Layer(Module):
    """
    nn.Linear + ReLU
    """
    def __init__(self, n_in, n_out, **kwargs):
        """Создает слой из `n_out` нейронов с `n_in` входами каждый."""
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]

    def __call__(self, x): # def forward(self);
        """Вычисляет выход слоя для входного вектора `x`.

        Возвращает один скаляр, если слой содержит один нейрон, иначе список.
        """
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        """Возвращает список параметров всех нейронов слоя."""
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

    def state_dict(self):
        """Возвращает состояние слоя как список состояний нейронов."""
        return [neuron.state_dict() for neuron in self.neurons]

    def load_state_dict(self, state_dict):
        """Загружает состояние слоя из списка состояний нейронов."""
        if len(state_dict) != len(self.neurons):
            raise ValueError(
                f"Expected {len(self.neurons)} neurons, got {len(state_dict)}"
            )

        for neuron, neuron_state in zip(self.neurons, state_dict):
            neuron.load_state_dict(neuron_state)

    def __repr__(self):
        """Возвращает строковое представление слоя и его нейронов."""
        return f"Layer({self.neurons})"


class MLP(Module):

    def __init__(self, n_in, n_outs):
        """Создает MLP с заданными размерностями слоев.

        `n_outs` задает числа нейронов в последовательно построенных слоях.
        """
        sizes = [n_in] + n_outs
        self.layers = [
            Layer(sizes[i], sizes[i + 1], nonlin=(i != len(n_outs) - 1))
            for i in range(len(n_outs))
        ]

    def __call__(self, x):
        """Пропускает вход `x` через все слои сети последовательно."""
        for i, layer in enumerate(self.layers):
            x = layer(x)

            # Если промежуточный слой вернул один скаляр,
            # следующий слой всё равно должен получить список входов.
            if i != len(self.layers) - 1 and isinstance(x, Value):
                x = [x]

        return x

    def parameters(self):
        """Возвращает параметры всех слоев MLP."""
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def state_dict(self):
        """Возвращает состояние MLP как список состояний слоев."""
        return [layer.state_dict() for layer in self.layers]

    def load_state_dict(self, state_dict):
        """Загружает состояние MLP из списка состояний слоев."""
        if len(state_dict) != len(self.layers):
            raise ValueError(
                f"Expected {len(self.layers)} layers, got {len(state_dict)}"
            )

        for layer, layer_state in zip(self.layers, state_dict):
            layer.load_state_dict(layer_state)

    def __repr__(self):
        """Возвращает строковое представление MLP и его слоев."""
        return f"MLP({self.layers})"

In [2]:
# Если вы все сделали правильно, должны пройти тесты

import torch
from torch import nn


class TorchMLP(nn.Module):
    def __init__(self, n_in, hidden_dims):
        super().__init__()
        if len(hidden_dims) != 2:
            raise ValueError("hidden_dims должен содержать ровно два размера")
        h1, h2 = hidden_dims
        self.fc1 = nn.Linear(n_in, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


def _copy_micro_to_torch(micro_mlp, torch_mlp):
    micro_state = micro_mlp.state_dict()
    torch_layers = [torch_mlp.fc1, torch_mlp.fc2, torch_mlp.fc3]
    for layer_state, torch_layer in zip(micro_state, torch_layers):
        w = torch.tensor([n["w"] for n in layer_state], dtype=torch.float32)
        b = torch.tensor([n["b"] for n in layer_state], dtype=torch.float32)
        with torch.no_grad():
            torch_layer.weight.copy_(w)
            torch_layer.bias.copy_(b)


def test_torch_mlp_equivalence():
    # Тест сравнивает micrograd-реализацию и torch-репликацию по выходам и градиентам
    n_in = 3
    hidden_dims = [4, 5]
    n_trials = 5

    for _ in range(n_trials):
        micro_mlp = MLP(n_in, hidden_dims + [1])
        torch_mlp = TorchMLP(n_in, hidden_dims)
        _copy_micro_to_torch(micro_mlp, torch_mlp)

        x_values = [random.uniform(-1, 1) for _ in range(n_in)]
        x_micro = [Value(v) for v in x_values]
        x_torch = torch.tensor(x_values, dtype=torch.float32, requires_grad=True)

        micro_mlp.zero_grad()
        torch_mlp.zero_grad()

        y_micro = micro_mlp(x_micro)
        y_torch = torch_mlp(x_torch)

        y_micro.backward()
        y_torch.backward()

        assert abs(y_micro.data - y_torch.item()) < 1e-6

        torch_layers = [torch_mlp.fc1, torch_mlp.fc2, torch_mlp.fc3]
        for layer_idx, micro_layer in enumerate(micro_mlp.layers):
            torch_layer = torch_layers[layer_idx]
            for neuron_idx, micro_neuron in enumerate(micro_layer.neurons):
                for w_idx, w in enumerate(micro_neuron.w):
                    torch_grad = torch_layer.weight.grad[neuron_idx, w_idx].item()
                    assert abs(w.grad - torch_grad) < 1e-6
                b_grad = torch_layer.bias.grad[neuron_idx].item()
                assert abs(micro_neuron.b.grad - b_grad) < 1e-6


test_torch_mlp_equivalence()

print("OK")


OK
